# K-Nearest Neighbors (KNN)

KNN is a **lazy learner** — it stores all training data and classifies new points by a majority vote of their K nearest neighbours in feature space.

**Dataset:** Iris — classify flower species from petal and sepal measurements.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '.')
from knn import KNN
np.random.seed(42)
print("Imports complete")

## Load & Explore the Data

In [ ]:
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(class_names)}")

# Pairplot of two key features
fig, axes = plt.subplots(1, 2, figsize=(12,5))
colors = ['#e63946','#457b9d','#2a9d8f']
for ax, (fi, fj) in zip(axes, [(2,3),(0,1)]):
    for cls in range(3):
        mask = y == cls
        ax.scatter(X[mask,fi], X[mask,fj], color=colors[cls],
                   label=class_names[cls], alpha=0.8, edgecolors='white', linewidths=0.3, s=50)
    ax.set_xlabel(feature_names[fi]); ax.set_ylabel(feature_names[fj])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_title("Petal features")
axes[1].set_title("Sepal features")
plt.suptitle("Iris Dataset — Feature Exploration", y=1.02)
plt.tight_layout()
plt.show()

## Preprocess & Split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Choosing K — Accuracy vs K

In [ ]:
k_values = list(range(1, 26))
accuracies = []

for k in k_values:
    clf = KNN(k=k)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = np.mean(np.array(preds) == y_test)
    accuracies.append(acc)

best_k = k_values[accuracies.index(max(accuracies))]
plt.figure(figsize=(9,5))
plt.plot(k_values, accuracies, 'o-', color='steelblue', linewidth=2, markersize=6)
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Best K={best_k}')
plt.xlabel("K (number of neighbours)")
plt.ylabel("Test Accuracy")
plt.title("KNN — Accuracy vs K")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print(f"Best K={best_k}, Accuracy={max(accuracies):.4f}")

## Train Best Model & Evaluate

In [ ]:
clf = KNN(k=best_k)
clf.fit(X_train, y_train)
y_pred = np.array(clf.predict(X_test))
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy (K={best_k}): {accuracy:.4f}")

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred, target_names=class_names))

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (K={best_k})")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 2D Decision Boundary (Petal Features)

In [ ]:
# Use only petal features for 2D boundary
X2 = X_scaled[:, 2:]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.20, random_state=42, stratify=y)

clf2 = KNN(k=best_k)
clf2.fit(X2_tr, y2_tr)

h = 0.05
x_min, x_max = X2[:,0].min()-0.5, X2[:,0].max()+0.5
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.column_stack([xx.ravel(), yy.ravel()])
Z = np.array(clf2.predict(grid)).reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
for cls in range(3):
    mask = y2_te == cls
    plt.scatter(X2_te[mask,0], X2_te[mask,1], color=colors[cls],
                label=class_names[cls], edgecolors='black', linewidths=0.5, s=60)
plt.xlabel("Petal length (std)")
plt.ylabel("Petal width (std)")
plt.title(f"KNN Decision Boundary (K={best_k}) — Petal Features")
plt.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

- KNN is simple, non-parametric, and needs no explicit training step.
- **Small K** = complex, noisy boundary (low bias, high variance).
- **Large K** = smoother boundary (high bias, low variance).
- Feature scaling is essential — KNN uses Euclidean distance.
- KNN is O(n) at prediction time — slow for large datasets.
